In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import random
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
from scipy.stats import spearmanr
import seaborn as sns

## Download Data

In [ ]:
metablomics_data = pd.read_csv("./data/bruna/data_PA/metabolomics_raw.tsv", delimiter='\t')
hematocrit_data = pd.read_csv("./data/shriya/SHMOLLI-output-unet-myocardium/hemocrit_only_trimmed.txt", sep='\t')
#all_phenotypes_regressed = pd.read_csv("./data/bruna/nature_revision/features_nature.tsv.qnorm.txt", sep='\t')
#abdonminal_phenotypes_data = pd.read_csv("./data/roger/brain_heart_PA/non_bmi_regressed/abdominal_merged.no_outliers.residuals.qnorm.txt", sep='\t')
cardiac_phenotype_data = pd.read_csv("./data/roger/brain_heart_PA/non_bmi_regressed/cardiac_merged2.no_outliers.residuals.qnorm.txt" , sep='\t')
#protien_data = pd.read_csv("./data/roger/brain_heart_PA/non_bmi_regressed/olink2_data_renamed.no_outliers.residuals.qnorm.txt", sep='\t')
#physical_activity_data = pd.read_csv("./data/roger/brain_heart_PA/non_bmi_regressed/PA_merged.no_outliers.residuals.qnorm.txt", sep='\t')

big_pheno_data = pd.read_csv("./data/bruna/lvedv/ukbb_all_no_exclusion_all_2_and_3_with_CMR_and_MR", sep='\t', header=0)

In [ ]:
unet_t1_regressed = pd.read_csv("./data/shriya/SHMOLLI-output-unet-myocardium-update2/cleaned_T1_percentiles_HHregressed_PHEWAS.no_outliers.residuals.qnorm.txt", sep='\t')
latent_dimensions = pd.read_csv("./data/shriya/SHMOLLI-output-unet-myocardium-update2/cleaned_latent_dimentions_PHEWAS.no_outliers.residuals.qnorm.txt", sep='\t')

#cardiac_sparce_PC_data = pd.read_csv("./data/roger/brain_heart_PA/non_bmi_regressed/cardiac_merged2_renamed_sparse_pca.tsv", sep='\t')

In [ ]:
metablomics_data.replace(-1000, np.nan, inplace=True)
#abdonminal_phenotypes_data.replace(-1000, np.nan, inplace=True)
hematocrit_data.replace(-1000, np.nan, inplace=True)
#protien_data.replace(-1000, np.nan, inplace=True)
#physical_activity_data.replace(-1000, np.nan, inplace=True)

big_pheno_data.replace(-1000, np.nan, inplace=True)

unet_t1_regressed.replace(-1000, np.nan, inplace=True)
latent_dimensions.replace(-1000, np.nan, inplace=True)

### All Merged Correllation Files

In [ ]:
all_metablomics_data = pd.merge(metablomics_data, hematocrit_data, left_on="FID", right_on="FID")
all_metablomics_data = all_metablomics_data.drop(columns = ["IID_x", "IID_y"])

In [ ]:
HH_regressed_columns = ['Mean_T1_regressed_hematocrit',
                    'T1_Standard_Deviation_regressed_hematocrit',
                    'T1_0.25th_Percentile_regressed_hematocrit', 
                    'T1_1th_Percentile_regressed_hematocrit',
                    'T1_25th_Percentile_regressed_hematocrit',
                    'T1_50th_Percentile_regressed_hematocrit',
                    'T1_75th_Percentile_regressed_hematocrit',
                    'T1_99th_Percentile_regressed_hematocrit',
                    'T1_99.75th_Percentile_regressed_hematocrit',
                  'Mean_T1_regressed_hematocrit_hypertension_status',
                    'T1_Standard_Deviation_regressed_hematocrit_hypertension_status',
                    'T1_0.25th_Percentile_regressed_hematocrit_hypertension_status', 
                    'T1_1th_Percentile_regressed_hematocrit_hypertension_status',
                    'T1_25th_Percentile_regressed_hematocrit_hypertension_status',
                    'T1_50th_Percentile_regressed_hematocrit_hypertension_status',
                    'T1_75th_Percentile_regressed_hematocrit_hypertension_status',
                    'T1_99th_Percentile_regressed_hematocrit_hypertension_status',
                    'T1_99.75th_Percentile_regressed_hematocrit_hypertension_status']

In [ ]:
unet_t1_regressed = unet_t1_regressed.drop(columns = HH_regressed_columns)

In [ ]:
unet_t1_all_metablomics_data = pd.merge(unet_t1_regressed, all_metablomics_data, left_on="IID", right_on="FID")
unet_t1_cardiac_phenotype_data = pd.merge(unet_t1_regressed, cardiac_phenotype_data, left_on="IID", right_on="id")
#unet_t1_abdonminal_phenotypes_data = pd.merge(unet_t1_regressed, abdonminal_phenotypes_data, left_on="IID", right_on="id")
#unet_t1_physical_activity_data = pd.merge(unet_t1_regressed, physical_activity_data, left_on="IID", right_on="id")

latent_all_metablomics_data = pd.merge(latent_dimensions, all_metablomics_data, left_on="FID", right_on="FID")
latent_cardiac_phenotype_data = pd.merge(latent_dimensions, cardiac_phenotype_data, left_on="FID", right_on="id")
#latent_abdonminal_phenotypes_data = pd.merge(latent_dimensions, abdonminal_phenotypes_data, left_on="FID", right_on="id")
#latent_physical_activity_data = pd.merge(latent_dimensions, physical_activity_data, left_on="FID", right_on="id")

In [ ]:
unet_t1_all_metablomics_data = unet_t1_all_metablomics_data.drop(columns = ["FID_x", "IID", "FID_y"])
unet_t1_cardiac_phenotype_data = unet_t1_cardiac_phenotype_data.drop(columns = ["FID", "IID", "id"])
#unet_t1_abdonminal_phenotypes_data = unet_t1_abdonminal_phenotypes_data.drop(columns = ["FID_x", "IID", "FID_y"])
#unet_t1_physical_activity_data = unet_t1_physical_activity_data.drop(columns = ["FID_x", "IID", "FID_y"])

#latent_all_metablomics_data = latent_all_metablomics_data.drop(columns = ["FID", "IID"])
#latent_cardiac_phenotype_data = latent_cardiac_phenotype_data.drop(columns = ["FID", "IID"])
#latent_abdonminal_phenotypes_data = latent_abdonminal_phenotypes_data.drop(columns = ["FID", "IID"])
#latent_physical_activity_data = latent_physical_activity_data.drop(columns = ["FID", "IID"])

In [ ]:
def plot_scatter(df, x_column, y_column, title=None, color='blue', alpha=0.7, size=50, figsize=(12, 10)):
    # Calculate correlations and skewness
    x_skewness = stats.skew(df[x_column])
    y_skewness = stats.skew(df[y_column])
    
    # Create figure with gridspec
    fig = plt.figure(figsize=figsize)
    grid = plt.GridSpec(4, 4, hspace=0.3, wspace=0.3)
    
    # Create main scatter plot
    ax_main = fig.add_subplot(grid[1:4, 0:3])
    scatter = ax_main.scatter(df[x_column], df[y_column], alpha=0.6)
    ax_main.set_xlabel(f"{x_column} (skewness: {x_skewness:.2f})")
    ax_main.set_ylabel(f"{y_column} (skewness: {y_skewness:.2f})")
    
    # Add a line of best fit for visualization
    x_values = np.linspace(df[x_column].min(), df[x_column].max(), 100)
    slope, intercept, r_value, p_value, std_err = stats.linregress(df[x_column], df[y_column])
    ax_main.plot(x_values, intercept + slope * x_values, 'r-', alpha=0.3)
    
    # Create x-histogram
    ax_histx = fig.add_subplot(grid[0, 0:3], sharex=ax_main)
    ax_histx.hist(df[x_column], bins=30, alpha=0.7)
    ax_histx.set_ylabel('Frequency')
    ax_histx.tick_params(axis="x", labelbottom=False)
    
    # Create y-histogram
    ax_histy = fig.add_subplot(grid[1:4, 3], sharey=ax_main)
    ax_histy.hist(df[y_column], bins=30, orientation='horizontal', alpha=0.7)
    ax_histy.set_xlabel('Frequency')
    ax_histy.tick_params(axis="y", labelleft=False)
    
    # Create info panel
    ax_info = fig.add_subplot(grid[0, 3])
    ax_info.axis('off')
    
    # Create a text box for correlation information
    info_text = (
#         f"Pearson r: {pearson_r:.3f} (p: {pearson_p:.3e})\n"
#         f"Spearman ρ: {spearman_r:.3f} (p: {spearman_p:.3e})\n\n"
#         f"Δ (Spearman-Pearson): {spearman_r-pearson_r:.3f}\n\n"
        f"X skewness: {x_skewness:.3f}\n"
        f"Y skewness: {y_skewness:.3f}"
    )
    
    # Add text box with correlation info
    ax_info.text(0.5, 0.5, info_text, ha='center', va='center', 
                 bbox=dict(boxstyle="round,pad=0.5", facecolor='white', alpha=0.8))
    
    plt.suptitle(f"Correlation Analysis: {y_column} vs {x_column}", fontsize=16)
    
    # Add an annotation indicating which correlation is better
#     if abs(spearman_r) > abs(pearson_r):
#         diff = abs(spearman_r) - abs(pearson_r)
#         ax_main.annotate(f"Spearman > Pearson by {diff:.3f}\nSuggesting non-linear or rank-based relationship",
#                          xy=(0.02, 0.98), xycoords='axes fraction',
#                          ha='left', va='top',
#                          bbox=dict(boxstyle="round,pad=0.3", fc="yellow", alpha=0.3))
    
    plt.tight_layout(rect=[0, 0, 1, 0.96])  # Adjust for the suptitle
    return fig

def show_corr_matrix(matrix):
    plt.figure(figsize=(10, 8))
    plt.imshow(matrix, cmap='coolwarm', interpolation='none', vmin=-1, vmax=1)
    plt.colorbar()
    plt.xticks(range(len(matrix.columns)), matrix.columns, rotation=90)
    plt.yticks(range(len(matrix.columns)), matrix.columns)
    plt.show()
    
def calculate_pvalues(data_frame):
    # Initialize matrices for p-values and correlation coefficients
    p_values_matrix = pd.DataFrame(index=data_frame.columns, columns=data_frame.columns)
    r_values_matrix = pd.DataFrame(index=data_frame.columns, columns=data_frame.columns)
    
    # Calculate Pearson correlation and p-values for each pair of variables
    for col1 in data_frame.columns:
        for col2 in data_frame.columns:
            # Extract values and drop any rows with NaN in either column
            vals1 = data_frame[col1].values
            vals2 = data_frame[col2].values
            mask = ~(np.isnan(vals1) | np.isnan(vals2))
            
            # Compute the Pearson correlation coefficient and p-value from the filtered values
            corr, p_value = pearsonr(vals1[mask], vals2[mask])
            
            p_values_matrix.loc[col1, col2] = p_value
            r_values_matrix.loc[col1, col2] = corr
    
    return p_values_matrix, r_values_matrix

def print_significant_correlations(p_values, r_values, threshold=0.05):
    # Create masks for significant p-values
    significant_mask = p_values < threshold
    
    # Get pairs of variables with significant correlations
    for i in range(len(p_values.columns)):
        for j in range(i + 1, len(p_values.columns)):  # i+1 to avoid duplicate pairs
            if significant_mask.iloc[i, j]:
                var1 = p_values.columns[i]
                var2 = p_values.columns[j]
                p_val = p_values.iloc[i, j]
                r_val = r_values.iloc[i, j]
                print(f"{var1} -- {var2}:")
                print(f"    r = {r_val:.3f}")
                print(f"    p = {p_val:.3e}\n")
                
def plot_significant_corr(p_value_df, r_values_matrix, name, threshold=0.05): # Define the T1 measures and metabolomic measures
    #t1_measures = [col for col in r_values_matrix.index if 'T1' in col]
    t1_measures = ['Mean_T1',
                    'T1_Standard_Deviation',
                    'T1_0.25th_Percentile', 
                    'T1_1th_Percentile',
                    'T1_25th_Percentile',
                    'T1_50th_Percentile',
                    'T1_75th_Percentile',
                    'T1_99th_Percentile',
                    'T1_99.75th_Percentile']

    metabolomic_measures = [col for col in r_values_matrix.columns 
                           if col not in t1_measures]
    
    # Extract and transpose the correlations
    r_values_subset = r_values_matrix.loc[t1_measures, metabolomic_measures].T
    p_values_subset = p_value_df.loc[t1_measures, metabolomic_measures].T
    
    # Convert to numeric and create mask for non-significant correlations
    r_values_numeric = r_values_subset.astype(float)
    p_values_numeric = p_values_subset.astype(float)
    mask = p_values_numeric > threshold
    
    # Create masked correlations
    masked_correlations = pd.DataFrame(
        np.where(mask, np.nan, r_values_numeric),
        index=metabolomic_measures,
        columns=t1_measures
    )
    
    # Set up the matplotlib figure (adjusted dimensions for transposed data)
    plt.figure(figsize=(16, 28))
    
    # Create heatmap
    ax = sns.heatmap(masked_correlations, 
                     annot=True,  # Show correlation values
                     mask=np.isnan(masked_correlations),  # Hide NaN values
                     cmap='RdBu_r',
                     center=0,
                     vmin=-1, 
                     vmax=1,
                     fmt='.3f',
                     cbar_kws={'label': 'Correlation Coefficient (r)'},
                     xticklabels=True,
                     yticklabels=True)
    
    # Rotate labels and adjust position
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    
    # Add title
    plt.title(f'Significant Correlation Adjusted Bonferroni (p = {threshold}) \nbetween T1 Values and {name}',
              pad=20, fontsize=14)
    
    # Adjust layout to prevent label cutoff
    plt.tight_layout()
    
    plt.show()
    
def plot_significant_latent_corr(p_value_df, r_values_matrix, name, threshold=0.05):
    # Define the Latent dimensions and metabolomic measures
    latent_measures = [col for col in r_values_matrix.index if 'Latent' in col]
    
    # If no latent measures found, check if they're named differently
    if not latent_measures:
        # Try to detect latent dimensions by pattern or just use numeric index columns
        latent_measures = [col for col in r_values_matrix.index if col.startswith('Latent_') or 'latent' in col.lower()]
    
    # If still empty, print column names to help debug
    if not latent_measures:
        print("Available columns:", r_values_matrix.index.tolist())
        return
    
    # Get metabolomic measures (everything that's not a latent measure)
    metabolomic_measures = [col for col in r_values_matrix.columns 
                           if col not in latent_measures]
    
    # Print some debug info
    print(f"Found {len(latent_measures)} latent measures and {len(metabolomic_measures)} metabolomic measures")
    
    # Check if we have data to plot
    if not latent_measures or not metabolomic_measures:
        print("Not enough data to plot. Make sure column names contain 'Latent'.")
        return
    
    # Extract and transpose the correlations
    r_values_subset = r_values_matrix.loc[latent_measures, metabolomic_measures].T
    p_values_subset = p_value_df.loc[latent_measures, metabolomic_measures].T
    
    # Convert to numeric and create mask for non-significant correlations
    r_values_numeric = r_values_subset.astype(float)
    p_values_numeric = p_values_subset.astype(float)
    mask = p_values_numeric > threshold
    
    # Create masked correlations
    masked_correlations = pd.DataFrame(
        np.where(mask, np.nan, r_values_numeric),
        index=metabolomic_measures,
        columns=latent_measures
    )
    
    # Check if we have any significant correlations
    if masked_correlations.notna().sum().sum() == 0:
        print("No significant correlations found with the current threshold.")
        return
    
    # Set up the matplotlib figure (adjusted dimensions for transposed data)
    plt.figure(figsize=(16, 8))
    
    # Create heatmap
    ax = sns.heatmap(masked_correlations, 
                     annot=True,  # Show correlation values
                     mask=np.isnan(masked_correlations),  # Hide NaN values
                     cmap='RdBu_r',
                     center=0,
                     vmin=-1, 
                     vmax=1,
                     fmt='.3f',
                     cbar_kws={'label': 'Correlation Coefficient (r)'},
                     xticklabels=True,
                     yticklabels=True)
    
    # Rotate labels and adjust position
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    
    # Add title
    plt.title(f'Significant Correlation Adjusted Bonferroni (p = {threshold}) \nbetween Latent Dimensions and {name}',
              pad=20, fontsize=14)
    
    # Adjust layout to prevent label cutoff
    plt.tight_layout()
    
    plt.show()

In [ ]:
plot_scatter(unet_t1_all_metablomics_data, 'Mean_T1', 'hematocrit')

In [ ]:
unet_t1_VAE_correlation_data = pd.merge(unet_t1_regressed, latent_dimensions, left_on="IID", right_on="FID")
unet_t1_VAE_correlation_data = unet_t1_VAE_correlation_data.drop(columns = ["FID_x", "IID_x", "FID_y", "IID_y"])
#unet_t1_VAE_correlation_data = unet_t1_VAE_correlation_data.drop(columns = HH_regressed_columns)

In [ ]:
unet_t1_VAE_correlation_matrix = unet_t1_VAE_correlation_data.corr()
show_corr_matrix(unet_t1_VAE_correlation_matrix)

In [ ]:
unet_t1_all_metablomics_data_correlation_matrix = unet_t1_all_metablomics_data.corr()
unet_t1_cardiac_phenotype_data_correlation_matrix = unet_t1_cardiac_phenotype_data.corr()
#unet_t1_abdonminal_phenotypes_data_correlation_matrix = unet_t1_abdonminal_phenotypes_data.corr()
#unet_t1_physical_activity_data_correlation_matrix = unet_t1_physical_activity_data.corr()

latent_all_metablomics_data_correlation_matrix = latent_all_metablomics_data.corr()
latent_cardiac_phenotype_data_correlation_matrix = latent_cardiac_phenotype_data.corr()
#latent_abdonminal_phenotypes_data_correlation_matrix = latent_abdonminal_phenotypes_data.corr()
#latent_physical_activity_data_correlation_matrix = latent_physical_activity_data.corr()

In [ ]:
show_corr_matrix(unet_t1_all_metablomics_data_correlation_matrix)

In [ ]:
show_corr_matrix(unet_t1_cardiac_phenotype_data_correlation_matrix)

In [ ]:
p_values_matrix, r_values_matrix = calculate_pvalues(unet_t1_VAE_correlation_data)
threshold = 0.05 / ((len(p_values_matrix) - 9 )*9)

plot_significant_corr(p_values_matrix, r_values_matrix, "UNET vs VAE Phenotypes", threshold = threshold)

In [ ]:
p_values_matrix, r_values_matrix = calculate_pvalues(unet_t1_all_metablomics_data)
threshold = 0.05 / ((len(p_values_matrix) - 9 )*9)

plot_significant_corr(p_values_matrix, r_values_matrix, "Metablomic Measures", threshold = threshold)

In [ ]:
p_values_matrix, r_values_matrix = calculate_pvalues(unet_t1_cardiac_phenotype_data)
threshold = 0.05 / ((len(p_values_matrix) - 9 )*9)

plot_significant_corr(p_values_matrix, r_values_matrix, "Cardiac Phenotypes", threshold = threshold)

## Spearman Correllations

In [ ]:
def calculate_spearman_pvalues(data_frame):
    p_values_matrix = pd.DataFrame(index=data_frame.columns, columns=data_frame.columns)
    r_values_matrix = pd.DataFrame(index=data_frame.columns, columns=data_frame.columns)
    
    # Calculate Spearman correlation and p-values for each pair of variables
    for col1 in data_frame.columns:
        for col2 in data_frame.columns:
            # Extract values and drop any rows with NaN in either column
            vals1 = data_frame[col1].values
            vals2 = data_frame[col2].values
            mask = ~(np.isnan(vals1) | np.isnan(vals2))
            
            # Compute the Spearman correlation coefficient and p-value from the filtered values
            corr, p_value = spearmanr(vals1[mask], vals2[mask])
            
            p_values_matrix.loc[col1, col2] = p_value
            r_values_matrix.loc[col1, col2] = corr
    
    return p_values_matrix, r_values_matrix

In [ ]:
p_values_matrix, r_values_matrix = calculate_spearman_pvalues(unet_t1_all_metablomics_data)
threshold = 0.05 / ((len(p_values_matrix) - 9 )*9)

plot_significant_corr(p_values_matrix, r_values_matrix, "Metablomic Measures", threshold = threshold)

## Correlation Compared to Latent Space Dimensions

In [ ]:
show_corr_matrix(latent_all_metablomics_data_correlation_matrix)

In [ ]:
show_corr_matrix(latent_cardiac_phenotype_data_correlation_matrix)

In [ ]:
p_values_matrix, r_values_matrix = calculate_pvalues(latent_cardiac_phenotype_data)
threshold = 0.05 / ((len(p_values_matrix) - 16 )*16)

plot_significant_latent_corr(p_values_matrix, r_values_matrix, "Cardiac Phenotypes", threshold = threshold)

In [ ]:
p_values_matrix, r_values_matrix = calculate_pvalues(latent_all_metablomics_data)
threshold = 0.05 / ((len(p_values_matrix) - 16 )*16)

plot_significant_latent_corr(p_values_matrix, r_values_matrix, "Metablomic Measures", threshold = threshold)

# Save Significant PHEWAS Tables

In [ ]:
def save_significant_corr(p_value_df, r_values_matrix, name, threshold=0.05, output_dir=''):
    """
    Save significant correlations between T1 measures and metabolomic measures to CSV.
    
    Parameters:
    -----------
    p_value_df : DataFrame
        P-values for correlations
    r_values_matrix : DataFrame
        Correlation coefficients
    name : str
        Name for the analysis (used in filename)
    threshold : float
        P-value threshold for significance (default: 0.05)
    output_dir : str
        Directory to save files (default: current directory)
    """
    # Define the T1 measures and metabolomic measures
    t1_measures = ['Mean_T1',
                    'T1_Standard_Deviation',
                    'T1_0.25th_Percentile', 
                    'T1_1th_Percentile',
                    'T1_25th_Percentile',
                    'T1_50th_Percentile',
                    'T1_75th_Percentile',
                    'T1_99th_Percentile',
                    'T1_99.75th_Percentile']

    metabolomic_measures = [col for col in r_values_matrix.columns 
                           if col not in t1_measures]
    
    # Extract and transpose the correlations
    r_values_subset = r_values_matrix.loc[t1_measures, metabolomic_measures].T
    p_values_subset = p_value_df.loc[t1_measures, metabolomic_measures].T
    
    # Convert to numeric and create mask for non-significant correlations
    r_values_numeric = r_values_subset.astype(float)
    p_values_numeric = p_values_subset.astype(float)
    mask = p_values_numeric > threshold
    
    # Create masked correlations (NaN for non-significant)
    masked_correlations = pd.DataFrame(
        np.where(mask, np.nan, r_values_numeric),
        index=metabolomic_measures,
        columns=t1_measures
    )
    
    # Create output path
    if output_dir and not output_dir.endswith('/'):
        output_dir += '/'
    
    # Save masked correlations
    corr_filename = f'{output_dir}significant_correlations_T1_{name}_p{threshold}.csv'
    masked_correlations.to_csv(corr_filename)
    print(f'Saved significant correlations to: {corr_filename}')
    
    # Optionally save p-values for significant correlations
    masked_pvalues = pd.DataFrame(
        np.where(mask, np.nan, p_values_numeric),
        index=metabolomic_measures,
        columns=t1_measures
    )
    pval_filename = f'{output_dir}pvalues_T1_{name}_p{threshold}.csv'
    masked_pvalues.to_csv(pval_filename)
    print(f'Saved p-values to: {pval_filename}')
    
    # Print summary
    n_significant = masked_correlations.notna().sum().sum()
    print(f'Found {n_significant} significant correlations at p < {threshold}')
    

def save_significant_latent_corr(p_value_df, r_values_matrix, name, threshold=0.05, output_dir=''):
    """
    Save significant correlations between Latent dimensions and metabolomic measures to CSV.
    
    Parameters:
    -----------
    p_value_df : DataFrame
        P-values for correlations
    r_values_matrix : DataFrame
        Correlation coefficients
    name : str
        Name for the analysis (used in filename)
    threshold : float
        P-value threshold for significance (default: 0.05)
    output_dir : str
        Directory to save files (default: current directory)
    """
    # Define the Latent dimensions and metabolomic measures
    latent_measures = [col for col in r_values_matrix.index if 'Latent' in col]
    
    # If no latent measures found, check if they're named differently
    if not latent_measures:
        latent_measures = [col for col in r_values_matrix.index if col.startswith('Latent_') or 'latent' in col.lower()]
    
    # If still empty, print column names to help debug
    if not latent_measures:
        print("Available columns:", r_values_matrix.index.tolist())
        print("ERROR: No latent measures found.")
        return
    
    # Get metabolomic measures (everything that's not a latent measure)
    metabolomic_measures = [col for col in r_values_matrix.columns 
                           if col not in latent_measures]
    
    # Print some debug info
    print(f"Found {len(latent_measures)} latent measures and {len(metabolomic_measures)} metabolomic measures")
    
    # Check if we have data to save
    if not latent_measures or not metabolomic_measures:
        print("ERROR: Not enough data to save. Make sure column names contain 'Latent'.")
        return
    
    # Extract and transpose the correlations
    r_values_subset = r_values_matrix.loc[latent_measures, metabolomic_measures].T
    p_values_subset = p_value_df.loc[latent_measures, metabolomic_measures].T
    
    # Convert to numeric and create mask for non-significant correlations
    r_values_numeric = r_values_subset.astype(float)
    p_values_numeric = p_values_subset.astype(float)
    mask = p_values_numeric > threshold
    
    # Create masked correlations
    masked_correlations = pd.DataFrame(
        np.where(mask, np.nan, r_values_numeric),
        index=metabolomic_measures,
        columns=latent_measures
    )
    
    # Check if we have any significant correlations
    n_significant = masked_correlations.notna().sum().sum()
    if n_significant == 0:
        print(f"WARNING: No significant correlations found with threshold p < {threshold}")
    
    # Create output path
    if output_dir and not output_dir.endswith('/'):
        output_dir += '/'
    
    # Save masked correlations
    corr_filename = f'{output_dir}significant_correlations_Latent_{name}_p{threshold}.csv'
    masked_correlations.to_csv(corr_filename)
    print(f'Saved significant correlations to: {corr_filename}')
    
    # Save p-values for significant correlations
    masked_pvalues = pd.DataFrame(
        np.where(mask, np.nan, p_values_numeric),
        index=metabolomic_measures,
        columns=latent_measures
    )
    pval_filename = f'{output_dir}pvalues_Latent_{name}_p{threshold}.csv'
    masked_pvalues.to_csv(pval_filename)
    print(f'Saved p-values to: {pval_filename}')
    
    # Print summary
    print(f'Found {n_significant} significant correlations at p < {threshold}')

In [ ]:
output_dir = './data/shriya/SHMOLLI-output-unet-myocardium-update2/'

p_values_matrix, r_values_matrix = calculate_pvalues(unet_t1_VAE_correlation_data)
threshold = 0.05 / ((len(p_values_matrix) - 9 )*9)
save_significant_corr(p_values_matrix, r_values_matrix, "UNET_vs_VAE_Phenotypes", threshold=threshold, output_dir=output_dir)

p_values_matrix, r_values_matrix = calculate_pvalues(unet_t1_all_metablomics_data)
threshold = 0.05 / ((len(p_values_matrix) - 9 )*9)
save_significant_corr(p_values_matrix, r_values_matrix, "Metablomic_Measures", threshold=threshold, output_dir=output_dir)

In [ ]:
output_dir = './data/shriya/SHMOLLI-output-unet-myocardium-update2/'

p_values_matrix, r_values_matrix = calculate_pvalues(unet_t1_cardiac_phenotype_data)
threshold = 0.05 / ((len(p_values_matrix) - 9 )*9)
save_significant_corr(p_values_matrix, r_values_matrix, "Cardiac_Phenotypes", threshold=threshold, output_dir=output_dir)

p_values_matrix, r_values_matrix = calculate_pvalues(latent_cardiac_phenotype_data)
threshold = 0.05 / ((len(p_values_matrix) - 16 )*16)
save_significant_latent_corr(p_values_matrix, r_values_matrix, "Cardiac_Phenotypes", threshold=threshold, output_dir=output_dir)

p_values_matrix, r_values_matrix = calculate_pvalues(latent_all_metablomics_data)
threshold = 0.05 / ((len(p_values_matrix) - 16 )*16)
save_significant_latent_corr(p_values_matrix, r_values_matrix, "Metablomic_Measures", threshold=threshold, output_dir=output_dir)